### Step 1 — Add a tiny logging utility (copy/paste once)

In [ ]:
import pandas as pd

class StepLogger:
    def __init__(self):
        self.records = []  # list of dicts

    def log(self, year: str, step: str, df: pd.DataFrame, note: str = ""):
        self.records.append({
            "year": year,
            "step": step,
            "rows": len(df),
            "note": note
        })

    def log_value(self, year: str, step: str, value: int, note: str = ""):
        self.records.append({
            "year": year,
            "step": step,
            "rows": int(value),
            "note": note
        })

    def to_counts_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.records)

def format_flow_table(counts_df: pd.DataFrame, steps_order: list[str]) -> pd.DataFrame:
    """
    Returns a report-ready table:
    index=Step, columns=Year, values="Removed: N | Pct% | Remaining: N"
    """
    out = pd.DataFrame(index=steps_order)

    for year in sorted(counts_df["year"].unique()):
        sub = counts_df[counts_df["year"] == year].set_index("step")["rows"]
        col = []
        prev = None

        for step in steps_order:
            n = int(sub.loc[step])
            if prev is None:
                col.append(f"Original: {n:,}")
            else:
                removed = prev - n
                pct = (removed / prev * 100) if prev else 0.0
                col.append(f"Removed: {removed:,} | {pct:.2f}% | Remaining: {n:,}")
            prev = n

        out[year] = col

    return out

### Step 2 — Define canonical step names (single source of truth)

In [ ]:
# --- Step naming (single source of truth) ---

STEPS_ORDER = [
    "Trips raw loaded",
    "Trips: remove WIND",
    "Status raw loaded",
    "Status: remove WIND",
    "Status: keep allowed event_types",
    "Merged status+trips (pre cor drop)",
    "Drop missing coordinates (cor)",
    "Drop trips duplicates <=60s",
    "Drop consecutive duplicates (same source)",
    "Drop consistency >30m (trip_end)",
    "Drop consistency >30m (trip_start)",
    "Drop consistency >65m around trip_end",
    "Final cleaned events",
]

KPI_STEPS = [
    "Parking assigned within 90m (KPI)"
]


### Step 3 — Add “allowed_event_types” and any mappings in one place

In [ ]:
# --- Cleaning configuration ---

ALLOWED_EVENT_TYPES = [
    "trip_end",
    "trip_start",
    "battery_low",
    "provider_drop_off",
    "maintenance_pick_up",
    "battery_charged",
    "agency_pick_up",
    "decommissioned",
    "maintenance",
    "rebalance_pick_up",
]

# If you have known normalizations in your notebook, centralize them here
EVENT_TYPE_REMAP = {
    "maintenance.non_operational": "maintenance",
    # add more remaps if your notebook has them
}


### Step 4.0 — Steplogger

In [ ]:
logger = StepLogger()


### Step 4 — Instrument ONE year first

In [ ]:
year = "2023"

In [ ]:
# ---- LOADS ----
logger.log(year, "Trips raw loaded", trips_check)
logger.log(year, "Status raw loaded", status_raw_check)

# ---- FILTER: remove WIND ----
trips_raw_filtered = trips_check[trips_check["provider_name"] != "WIND"].copy()
logger.log(year, "Trips: remove WIND", trips_raw_filtered)

status_raw_filtered = status_raw_check[status_raw_check["provider_name"] != "WIND"].copy()
logger.log(year, "Status: remove WIND", status_raw_filtered)

# ---- FILTER: allowed event_types (Status) ----
status_raw_filtered["event_types"] = status_raw_filtered["event_types"].replace(EVENT_TYPE_REMAP)
status_filtered = status_raw_filtered[status_raw_filtered["event_types"].isin(ALLOWED_EVENT_TYPES)].copy()
logger.log(year, "Status: keep allowed event_types", status_filtered)

# ---- MERGE (pre cor drop) ----
# (Use the DF that exists in your notebook at this stage)
logger.log(year, "Merged status+trips (pre cor drop)", merged_two_temps)

# ---- FILTER: drop missing coordinates ----
events_trips = merged_two_temps.dropna(subset=["cor"]).copy()
logger.log(year, "Drop missing coordinates (cor)", events_trips)

# ---- FILTER: duplicate-like drops <=60s ----
merged_filtered = merged_status_trips[~condition_to_drop].copy()
logger.log(year, "Drop trips duplicates <=60s", merged_filtered)

# ---- FILTER: consecutive duplicates ----
merged_filtered_without_dup = merged_filtered[~duplicate_drop].copy()
logger.log(year, "Drop consecutive duplicates (same source)", merged_filtered_without_dup)

# ---- FILTER: consistency rules (>30m) ----
filtered_event_types = merged_filtered_event_types[~trip_end_cons_rule].copy()
logger.log(year, "Drop consistency >30m (trip_end)", filtered_event_types)

filtered_event_types = filtered_event_types[~trip_start_cons_rule].copy()
logger.log(year, "Drop consistency >30m (trip_start)", filtered_event_types)

# ---- FILTER: consistency rules (>65m) ----
cleared_df = filtered_df[~cons_over65_to_remove].copy()
logger.log(year, "Drop consistency >65m around trip_end", cleared_df)

# ---- FINAL OUTPUT DF ----
logger.log(year, "Final cleaned events", merged_clear)

# ---- KPI: parking assignment (NOT a removal step) ----
assigned_n = int(merged_clear["parking_spot"].notna().sum())
total_n = int(len(merged_clear))
logger.log_value(year, "Parking assigned within 90m (KPI)", assigned_n, note=f"out of {total_n}")
